In [ ]:
import os

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from copy import deepcopy

import random

from pathlib import Path

SEED = 69
np.random.seed(SEED)
torch.manual_seed(SEED)

Array = np.ndarray


@dataclass
class SimulationConfig:
    nx = 16
    ny = 16
    pixel_size_m = 2e-6
    z_length_m = 3e-6
    z_speed_um_s = 9.0
    sampling_frequency_hz = 9000
    setpoint_force_n = 1.0e-9
    tip_shape = "Sphere"
    include_thermal_noise = True
    noise_type = "experimental"
    experimental_noise_path = \
      "thermal-noise-data_vDeflection_2025.11.28-16.42.57.tnd"
    curve_mode = "approach_retract"  # options: "approach", "approach_retract"

@dataclass
class DatasetConfig:
    n_train_samples = 20
    n_val_samples = 15
    n_test_samples = 10
    topography_types = ("gaussian_features", "hemisphere")
    substrate_stiffness_range_pa = (1e3, 5e3)
    feature_stiffness_range_pa = (0.1e3, 5e3)
    height_range_m = (0.1e-6, 1.5e-6)
    contact_point_range_m = (0.01e-6, 2.9e-6)

@dataclass
class ModelConfig:
    batch_size = 256
    learning_rate = 2e-3
    max_epochs =10
    predict_contact_point = True
    cp_loss_weight = 0.5
    num_workers = 0
    accelerator = "auto"
    devices = 1
    default_sample_to_display = 0


simulation_config = SimulationConfig()
dataset_config = DatasetConfig()
model_config = ModelConfig()

print(simulation_config)
print(dataset_config)
print(model_config)

class HertzContact:
    """Hertz contact model for spherical and pyramidal tips.

    Represents the relationship between indentation and force. Stores
    information related to contact geometry and contact mechanics.

    Parameters
    ----------
    R: float
        Tip radius in meters.
    nu: float
        Poisson ratio of the sample.
    alpha: float
        Pyramid face angle in degrees.

    """

    def __init__(
        self,
        R: float = 10e-9,
        nu: float = 0.5,
        alpha: float = 18.0
    ) -> None:
        """Initializes the HertzContact model.

        Initializes the HertzContact model with given parameters.

        Parameters
        ----------
        R: float
            Tip radius in meters.
        nu: float
            Poisson ratio of the sample.
        alpha: float
            Pyramid face angle in degrees.

        """

        self.R = R
        self.nu = nu
        self.alpha = alpha

    @property
    def alpha_rad(
        self
    ) -> float:
        """Return the pyramid face angle in radians.

        Returns
        -------
        float
            Pyramid face angle in radians.

        """

        return np.deg2rad(self.alpha)

    def hertz_sphere(
        self,
        indentation_m: float,
        stiffness_pa: float
    ) -> float:
        """Return the Hertz force for a spherical tip.

        Returns the force response of a purely elastic semi finite substrate
        with the given stiffness in response to given indentation depth for a
        spherical indentor.

        Parameters
        ----------
        indentation_m : float
            Indentation depth in meters.
        stiffness_pa : float
            Young's modulus in pascal.

        Returns
        -------
        float
            Contact force in newtons.

        """

        return (
            (4.0 / 3.0)
            * stiffness_pa
            * np.sqrt(self.R)
            * indentation_m**1.5
            / (1.0 - self.nu**2)
        )

    def hertz_cone(
        self,
        indentation_m: float,
        stiffness_pa: float
    ) -> float:
        """Return the Hertz-like force for a conical approximation.

        Returns the force response of a purely elastic semi finite substrate
        with the given stiffness in response to given indentation depth for a
        conical indentor.

        Parameters
        ----------
        indentation_m : float
            Indentation depth in meters.
        stiffness_pa : float
            Young's modulus in pascal.

        Returns
        -------
        float
            Contact force in newtons.

        """

        return (
            stiffness_pa
            * np.tan(self.alpha_rad)
            * indentation_m**2
            / (np.sqrt(2.0) * (1.0 - self.nu**2))
        )

    def force(
        self,
        cantilever_shape: str,
        indentation_m: float,
        stiffness_pa: float
    ) -> float:
        """Return the contact force for the given indentation.

        Parameters
        ----------
        cantilever_shape: str
            Tip geometry, either ``'Sphere'`` or ``'Pyramid'``.
        indentation_m: float
            Indentation depth in meters.
        stiffness_pa: float
            Young's modulus in pascal.

        Returns
        -------
        float
            Contact force in newtons.

        """

        indentation_safe_m = \
          np.maximum(np.asarray(indentation_m, dtype=np.float64), 0.0)
        if cantilever_shape == "Sphere":
            return self.hertz_sphere(indentation_safe_m, stiffness_pa)
        if cantilever_shape == "Pyramid":
            return self.hertz_cone(indentation_safe_m, stiffness_pa)
        raise ValueError(f"Unsupported cantilever shape: {cantilever_shape}")


class AFMCantilever:
    """AFM cantilever model with thermal-noise PSD support.

    This class represents the cantilever and stores parameters relevant to its
    thermal noise characteristics, allowing for both theoretical and
    experimental PSDs to be used in simulations.

    Parameters
    ----------
    k : float
        Spring constant in N/m.
    f0 : float
        Resonance frequency in Hz.
    Q : float
        Quality factor.
    T : float
        Temperature in kelvin.

    """

    def __init__(
        self,
        k: float =0.1,
        f0: float =7e3,
        Q: float =10.0,
        T: float =300.0
    ) -> None:
        """Initializes the AFMCantilever.

        Initializes the AFMCantilever with given parameters and computes
        derived properties.

        Parameters
        ----------
        k : float, optional
            Spring constant in N/m (default is 0.1).
        f0 : float, optional
            Resonance frequency in Hz (default is 7e3).
        Q : float, optional
            Quality factor (default is 10.0).
        T : float, optional
            Temperature in kelvin (default is 300.0).

        """

        self.k = k
        self.f0 = f0
        self.Q = Q
        self.T = T
        self.kB = 1.380649e-23
        self.m_eff = k / (2.0 * np.pi * f0) ** 2
        self.gamma = 2.0 * np.pi * f0 * self.m_eff / Q
        self._experimental_psd_cache = {}
        self._grid_psd_cache = {}

    def thermal_noise_rms(
            self
        ) -> float:
        """Return the RMS thermal deflection in meters.

        Returns
        -------
        float
            RMS thermal deflection in meters.

        """

        return np.sqrt(self.kB * self.T / self.k)

    def thermal_noise_psd_theoretical(
        self,
        frequency_hz: np.ndarray
    ) -> np.ndarray:
        """Return the one-sided theoretical displacement PSD.

        Parameters
        ----------
        frequency_hz : np.ndarray
                Frequencies in Hz at which to evaluate the PSD.

        Returns
        -------
        np.ndarray
            PSD in m^2/Hz.

        """

        omega  = 2.0 * np.pi * frequency_hz
        omega0 = 2.0 * np.pi * self.f0
        denom  = (omega0**2 - omega**2)**2 + (omega * omega0 / self.Q)**2
        return (4.0 * self.kB * self.T * self.gamma) / (self.k**2 * denom)

    def thermal_noise_psd_experimental(
        self,
        path: str
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Load and cache an experimental displacement PSD.

        Loads the experimental PSD from the provided path. Converts the
        displacement from volts to meters using the metadata in the file.

        Parameters
        ----------
        path : str
            Path to the experimental PSD file.

        Returns
        -------
        tuple of np.ndarray
            (frequency_hz, spectral_density_x, spectral_density_x_fit)

        """

        cache_key = str(path)
        if cache_key not in self._experimental_psd_cache:
            data = np.loadtxt(path, skiprows=24)
            frequency_hz       = data[:, 0]
            spectral_density_v = data[:, 1]
            spectral_density_v_fit = data[:, 3]
            with open(path, "r") as f:
                lines = f.readlines()
            for line in lines:
                if "sensitivity" in line:
                    volts_to_displacement_nm = \
                      float(line.split(":")[1].split()[0])
                if "parameter.f" in line:
                    self.f0 = float(line.split(":")[1].split()[0]) * 1e3
                if "parameter.Q" in line:
                    self.Q = float(line.split(":")[1].split()[0])
            scale = volts_to_displacement_nm * 1e-9
            self._experimental_psd_cache[cache_key] = (
                frequency_hz,
                spectral_density_v     * scale**2,
                spectral_density_v_fit * scale**2,
            )
        return self._experimental_psd_cache[cache_key]

    def get_noise_psd(
        self,
        n_points: int,
        sampling_frequency_hz: float,
        noise_type: str = simulation_config.noise_type,
        experimental_path: str = simulation_config.experimental_noise_path
    ) -> tuple[np.ndarray, np.ndarray]:
        """Returns the simulation-grid PSD.

        Returns the theoretical PSD calculated with the harmonic oscillator
        model. When it's possible the cached results are reused.

        Parameters
        ----------
        n_points : int
            Number of points in the simulation grid.
        sampling_frequency_hz : float
            Sampling frequency in Hz.
        noise_type : str
            Type of noise ("theoretical", "experimental" or "rms").
        experimental_path : str or None
            Path to the experimental PSD file.

        Returns
        -------
        tuple of np.ndarray
            (frequency_hz, spectral_density_x)

        """

        cache_key = (noise_type.lower(), int(n_points), \
                     float(sampling_frequency_hz),None if experimental_path
                     is None else str(experimental_path))
        if cache_key in self._grid_psd_cache:
            return self._grid_psd_cache[cache_key]

        frequency_hz = np.fft.rfftfreq(n_points, d=1.0 / sampling_frequency_hz)
        if noise_type.lower() == "theoretical":
            spectral_density_x = \
              self.thermal_noise_psd_theoretical(frequency_hz)
        elif noise_type.lower() == "experimental":
            if experimental_path is None:
                raise ValueError("`experimental_path` must be provided for \
                experimental PSD.")
            frequency_hz, spectral_density_x, _ = (
                self.thermal_noise_psd_experimental(experimental_path)
            )
        else:
            raise ValueError(f"Unknown noise type: {noise_type}")

        self._grid_psd_cache[cache_key] = (frequency_hz, spectral_density_x)
        return self._grid_psd_cache[cache_key]

    def sample_thermal_noise(
        self,
        n_points: int,
        sampling_frequency_hz: float,
        noise_type: str = simulation_config.noise_type,
        experimental_path: str = simulation_config.experimental_noise_path,
        rng=None
    ) -> np.ndarray:
        """Draws one thermal-noise trace from the PSD.

        Draws one trace of thermal noise from the PSD using random phase and
        converts it to time domain noise using np.fft.irfft

        Parameters
        ----------
        n_points : int
            Number of points in the simulation grid.
        sampling_frequency_hz : float
            Sampling frequency in Hz.
        noise_type : str
            Type of noise ("theoretical", "experimental" or "rms").
        experimental_path : str or None
            Path to the experimental PSD file.
        rng : numpy.random.Generator or None
            Random number generator instance. If None, a new instance is
            created.

        Returns
        -------
        np.ndarray
            Noise trace in meters, shape (n_points,).

        """

        rng = np.random.default_rng() if rng is None else rng
        frequency_hz, spectral_density_x = self.get_noise_psd(
            n_points, sampling_frequency_hz, noise_type, experimental_path
        )
        df        = sampling_frequency_hz / n_points

        amplitude = n_points * \
                            np.sqrt(np.maximum(spectral_density_x * df/2, 0.0))
        coefficients = np.zeros(amplitude.shape[0], dtype=np.complex128)

        if amplitude.shape[0] > 2:
            phase = rng.uniform(0.0, 2.0 * np.pi, size=amplitude.shape[0] - 2)
            coefficients[1:-1] = amplitude[1:-1] * np.exp(1j * phase)
        coefficients[0] = 0.0
        if n_points % 2 == 0 and amplitude.shape[0] > 1:
            coefficients[-1] = amplitude[-1] * np.sign(rng.standard_normal())

        noise_m = np.fft.irfft(coefficients, n=n_points).real
        rms      = self.thermal_noise_rms()
        std      = np.std(noise_m)
        # Guard against zero std.
        if std > 0:
            noise_m = noise_m / std * rms
        else:
            noise_m = rng.normal(scale=rms, size=n_points)
        return noise_m

cantilever = AFMCantilever(k=0.178, f0=10e3, Q=100.0, T=300.0)
contact_model = HertzContact(R=1e-6, nu=0.5, alpha=18.0)

cantilever=cantilever
contact_model=contact_model
config=simulation_config

def _compute_n_points_approach(
) -> int:
    """ Compute the number of points for the approach phase

    Computes the number of points for the approach phasebased on the z-length,
    z-speed, and sampling frequency.

    Returns
    -------
    int
        Number of points for the approach phase, at least 32.

    """

    total_time_s = config.z_length_m / \
      (config.z_speed_um_s * 1e-6)
    return max(32, int(np.ceil(total_time_s * \
                                config.sampling_frequency_hz)))

def _build_approach_axis(
) -> np.ndarray:
    """Build the approach displacement axis.

    Builds the approach displacement axis as a linearly spaced array from 0 to z_length_m with n_points_approach points.

    Returns
    -------
    np.ndarray
        Approach displacement axis in meters, shape (n_points_approach,).

    """

    return np.linspace(0.0, config.z_length_m, \
                       n_points_approach, \
                        endpoint=True, dtype=np.float64)

def _retract_axis(
    z_start_m: float
) -> np.ndarray:
    """Return the retract displacement axis

    Returns the retract displacement axis starting from ``z_start_m``.

    Parameters
    ----------
    z_start_m : float
        Starting z-position for the retract phase in meters.

    Returns
    -------
    np.ndarray
        Retract displacement axis in meters.

    """

    z_stop_m = max(0.0, z_start_m - config.z_length_m)
    if z_stop_m >= z_start_m:
        return np.zeros(0, dtype=np.float64)
    n = max(1, int(np.ceil(
        (z_start_m - z_stop_m) / (config.z_speed_um_s * 1e-6) \
        * config.sampling_frequency_hz
    )))
    return np.linspace(z_start_m, z_stop_m, n + 1, \
                       endpoint=True)[1:].astype(np.float64)


In [ ]:
stiffness_range_pa = (0.1e3, 5e3)
contact_point_range_m = (0.01e-6, 2.9e-6)
align_length = 1500
rng = np.random.default_rng()

data_x1 = []
data_y1 = []

data_x2 = []
data_y2 = []

for i in range(8192):

    n_points_approach = _compute_n_points_approach()
    approach_axis_m   = _build_approach_axis()

    stiffness_pa = rng.uniform(stiffness_range_pa[0], stiffness_range_pa[1])
    contact_point_m = rng.uniform(contact_point_range_m[0], contact_point_range_m[1])
    
    approach_indent = np.maximum(
    approach_axis_m - contact_point_m, 0.0
    )
    approach_force = contact_model.force(
        cantilever_shape=config.tip_shape,
        indentation_m=approach_indent,
        stiffness_pa=stiffness_pa,
    )
    
    exceeded = approach_force >= config.setpoint_force_n
    
    cp    = float(contact_point_m)
    E     = float(stiffness_pa)


    contact_indices = np.clip(
        np.searchsorted(approach_axis_m, contact_point_m,
                        side="left"),
        0, n_points_approach - 1,
    )

    # ── approach ─────────────────────────────────────────────────
    if True in exceeded:
        ei = first_exceed   = np.argmax(exceeded)
    
        app_z = approach_axis_m[:ei]
        app_f = approach_force[:ei]
        
    else:

        app_z = approach_axis_m.copy()
        app_f = approach_force.copy()
    
    # ── retract ──────────────────────────────────────────────────
    ret_z = _retract_axis(float(app_z[-1]))

    ret_f = contact_model.force(
        config.tip_shape,
        np.maximum(ret_z - cp, 0.0),
        E,
    )

    cp_idx = int(contact_indices)
    tip_app_ind = app_z[cp_idx:] - \
    app_f[cp_idx:] / cantilever.k
    tip_app = np.concatenate([app_z[:cp_idx], tip_app_ind])
    tip_ret_ind = ret_z[cp_idx:] - \
    ret_f[cp_idx:] / cantilever.k
    tip_ret = np.concatenate([ret_z[:cp_idx], tip_ret_ind])

    disp_z  = np.concatenate([app_z, ret_z])
    disp_tip  = np.concatenate([tip_app, tip_ret])
    force_n = np.concatenate([app_f, ret_f])
    retract_start = app_z.size
    
    # ── thermal noise ────────────────────────────────────────────
    noise_m = np.zeros_like(force_n)
    if True:
        if simulation_config.noise_type == "rms":
            sigma = cantilever.thermal_noise_rms()
            noise_m += np.random.normal(-sigma, sigma,
                                        size=len(noise_m))
        else:
            noise_m = cantilever.sample_thermal_noise(
                n_points=disp_z.size,
                sampling_frequency_hz= \
                  config.sampling_frequency_hz,
                noise_type=simulation_config.noise_type,
                experimental_path=simulation_config.experimental_noise_path,
                rng=rng,
            )

    displacement_curves_m   = disp_z
    displacement_tip_curves_m   = disp_tip
    force_curves_n         = force_n
    measured_force_curves_n = force_n + noise_m * cantilever.k
    curve_lengths           = disp_z.size
    retract_start_indices   = retract_start

    for j in range(10):
    
        cp = deepcopy(contact_point_m)
        stif = deepcopy(stiffness_pa)
        
        disp = tip_app.copy()
        meas = measured_force_curves_n[:tip_app.shape[0]].copy()

        cp = disp.max() - cp
        disp = disp.max() - disp
        cp = cp/disp.max()

        cut1 = int(random.random()*0.5*cp*disp.size)
        cut2 = int((1-(1-cp)*random.random()*0.9)*disp.size)

        cp = cp*disp.max()

        disp = disp[::-1][cut1:cut2]
        meas = meas[::-1][cut1:cut2]

 
        x_old = np.linspace(0, 1, disp.size)
        x_new = np.linspace(0, 1, align_length)
    
        disp = np.interp(x_new, x_old, disp)
        meas = np.interp(x_new, x_old, meas)
    

        cp = cp - disp.min()
        disp = disp - disp.min()
        cp = cp/disp.max()

        disp = disp[::-1]
        meas = meas[::-1]
        
    
        data_x1.append(np.hstack([disp*1e6,meas*1e9]))
        data_y1.append([cp])



        cp = deepcopy(contact_point_m)
        stif = deepcopy(stiffness_pa)
        
        disp = tip_app.copy()
        meas = measured_force_curves_n[:tip_app.shape[0]].copy()

        cp = disp.max() - cp
        disp = disp.max() - disp
        cp = cp/disp.max()

        cut1 = int(random.random()*0.5*cp*disp.size)
        cut2 = int((1-(1-cp)*(0.95+0.05*random.random()))*disp.size)

        cp = cp*disp.max()

        disp = disp[::-1][cut1:cut2]
        meas = meas[::-1][cut1:cut2]

 
        x_old = np.linspace(0, 1, disp.size)
        x_new = np.linspace(0, 1, align_length)
    
        disp = np.interp(x_new, x_old, disp)
        meas = np.interp(x_new, x_old, meas)
    

        cp = cp - disp.min()
        disp = disp - disp.min()
        cp = cp/disp.max()

        disp = disp[::-1]
        meas = meas[::-1]
        
    
        data_x2.append(np.hstack([disp*1e6,meas*1e9]))
        data_y2.append([stif])

## cp_model

In [ ]:
data_x_r = np.array(data_x1)
data_y_r = np.array(data_y1)
data_y_r_n = data_y_r

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


x_train, x_test, y_train, y_test = train_test_split(
    data_x_r, data_y_r_n,
    test_size=0.3,
    random_state=577
)

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=4096)
val_loader = DataLoader(val_dataset, batch_size=4096)

In [ ]:
import deeplay as dl

mlp_model = dl.MultiLayerPerceptron(
    in_features=3000, hidden_features=[1024, 256], out_features=1,
).create()

print(mlp_model)

from torch.nn import MSELoss as MSE
from torchmetrics import MeanAbsoluteError as MAE

regressor = dl.Regressor(
    mlp_model, loss=MSE(), optimizer=dl.Adam(lr=1e-4),
).create()

print(regressor)

In [ ]:
import deeplay as dl
from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

# Save the top-1 model according to the validation loss
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=5,
)

# Set the logger
csv_logger = CSVLogger("logs", name="regressor")

# Instantiate the trainer
trainer = dl.Trainer(
    max_epochs=5000,
    accelerator="auto",
    callbacks=[checkpoint_callback],
    logger=csv_logger,
    log_every_n_steps=5,
    enable_progress_bar=False,
)

In [ ]:
trainer.fit(regressor, train_loader, val_loader)

In [ ]:
print(f"The best model is saved at {checkpoint_callback.best_model_path}")

best_cp_model = dl.Regressor.load_from_checkpoint(
    checkpoint_callback.best_model_path
)

In [ ]:
best_cp_model = dl.Regressor.load_from_checkpoint(
    "./models/epoch=1403-step=19656.ckpt"
)

In [ ]:
trainer.test(best_cp_model, val_loader)

In [ ]:
best_model.eval()

all_y = []
all_y_hat = []
all_x = []
with torch.no_grad():
    for batch in val_loader:
        x, y = batch


        y_hat = best_cp_model(x)
        all_x.append(x)
        all_y.append(y)
        all_y_hat.append(y_hat)

all_x = torch.cat(all_x, dim=0)
all_y = torch.cat(all_y, dim=0)
all_y_hat = torch.cat(all_y_hat, dim=0)

In [ ]:
all_x = all_x.cpu().numpy()
all_y = all_y.cpu().numpy()
all_y_hat = all_y_hat.cpu().numpy()

In [ ]:
plt.scatter(all_y[:,0]*all_x[:,0:1500].max(axis=1), all_y_hat[:,0]*all_x[:,0:1500].max(axis=1))

In [ ]:
from sklearn.metrics import root_mean_squared_error

In [ ]:
root_mean_squared_error(all_y[:,0]*all_x[:,0:1500].max(axis=1), all_y_hat[:,0]*all_x[:,0:1500].max(axis=1))*1e3

In [ ]:
cut_percentage = 0


import os

if not os.path.exists("3t3_cell_dataset"):
    os.system("git clone https://github.com/DeepTrackAI/3t3_cell_dataset")

from pathlib import Path
import numpy as np

# Load the approach curves
approach_path = Path.cwd() / "3t3_cell_dataset" / "approach"
approach_files = sorted(approach_path.glob("*.npy"), key=lambda f: int(f.stem))
approach_curves = [np.load(f) for f in approach_files]

# Load the retraction curves
retract_path = Path.cwd() / "3t3_cell_dataset" / "retraction"
retract_files = sorted(retract_path.glob("*.npy"), key=lambda f: int(f.stem))
retract_curves = [np.load(f) for f in retract_files]

# Load the labels
label_file = Path.cwd() / "3t3_cell_dataset" / "label.npy"
labels = np.load(label_file)

# Load the contact points
contactpoint_file = Path.cwd() / "3t3_cell_dataset" / "contact_point.npy"
contactpoints = np.load(contactpoint_file)

# Load the contact points
young_modulus_file = Path.cwd() / "3t3_cell_dataset" / "young_modulus.npy"
young_modulus = np.load(young_modulus_file)

# Calculate the length of each curve
approach_curve_lengths = [curve.shape[0] for curve in approach_curves]
retract_curve_lengths = [curve.shape[0] for curve in retract_curves]

# Calculate the minimum length of approach curves and retraction curves
min_approach_curve_length = min(approach_curve_lengths)
min_retract_curve_length = min(retract_curve_lengths)

print(
    f"Minimum approach curve length = {min_approach_curve_length}\n"
    f"Minimum retraction curve length = {min_retract_curve_length}\n"
)


data_exp_x1 = []
data_exp_x2 = []

for i in approach_curves:
    data_exp_x1.append(i[int(i.shape[0]*cut_percentage)::,0])
    data_exp_x2.append(i[int(i.shape[0]*cut_percentage)::,1])

import numpy as np

def resample_to_same_length(data_list, target_len):
    
    resampled = []

    for data in data_list:
        data = np.asarray(data)
        original_len = len(data)

        x_old = np.linspace(0, 1, original_len)

        x_new = np.linspace(0, 1, target_len)

        y_new = np.interp(x_new, x_old, data)

        resampled.append(y_new)

    return resampled

teste1 = resample_to_same_length(data_exp_x1, 1500)
teste2 = resample_to_same_length(data_exp_x2, 1500)

teste1 = np.array(teste1)
teste2 = np.array(teste2)

contactpoints =  contactpoints - teste1.min(axis=1)
teste1 = teste1 - teste1.min(axis=1).reshape(-1,1)
contactpoints =  contactpoints/teste1.max(axis=1)
data_xe_r = np.hstack([teste1, teste2])
data_ye_r = np.vstack([contactpoints]).T

real_dataset = TensorDataset(
    torch.tensor(data_xe_r, dtype=torch.float32),
    torch.tensor(data_ye_r, dtype=torch.float32)
)


exp_loader = DataLoader(real_dataset, batch_size=data_xe_r.shape[0])

In [ ]:
trainer.test(best_cp_model, exp_loader)

In [ ]:
best_cp_model.eval()

all_y = []
all_y_hat = []
all_x = []
with torch.no_grad():
    for batch in exp_loader:
        x, y = batch

        y_hat = best_cp_model(x)
        all_x.append(x)
        all_y.append(y)
        all_y_hat.append(y_hat)

all_x = torch.cat(all_x, dim=0)
all_y = torch.cat(all_y, dim=0)
all_y_hat = torch.cat(all_y_hat, dim=0)

all_x = all_x.cpu().numpy()
all_y = all_y.cpu().numpy()
all_y_hat = all_y_hat.cpu().numpy()

In [ ]:
plt.scatter((all_y[:,0]*all_x[:,0:1500].max(axis=1))[labels==1], (all_y_hat[:,0]*all_x[:,0:1500].max(axis=1))[labels==1])

In [ ]:
root_mean_squared_error((all_y[:,0]*all_x[:,0:1500].max(axis=1))[labels==1], (all_y_hat[:,0]*all_x[:,0:1500].max(axis=1))[labels==1])*1e3

In [ ]:
cp_pred_percentage = all_y_hat[:,0].copy()

In [ ]:
cp_pred_real = (all_y_hat[:,0]*all_x[:,0:1500].max(axis=1)).copy()

## stiffness model

In [ ]:
data_x_r = np.array(data_x2)
data_y_r = np.array(data_y2)
data_y_r[:,0] = (data_y_r[:,0] - stiffness_range_pa[0])/(stiffness_range_pa[1] - stiffness_range_pa[0])
data_y_r_n = data_y_r

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


x_train, x_test, y_train, y_test = train_test_split(
    data_x_r, data_y_r_n,
    test_size=0.3,
    random_state=577
)

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=4096)
val_loader = DataLoader(val_dataset, batch_size=4096)

In [ ]:
import deeplay as dl

mlp_model = dl.MultiLayerPerceptron(
    in_features=3000, hidden_features=[1024, 256], out_features=1,
).create()

print(mlp_model)

from torch.nn import MSELoss as MSE
from torchmetrics import MeanAbsoluteError as MAE

regressor = dl.Regressor(
    mlp_model, loss=MSE(), optimizer=dl.Adam(lr=1e-4),
).create()

print(regressor)

In [ ]:
import deeplay as dl
from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

# Save the top-1 model according to the validation loss
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=5,
)

# Set the logger
csv_logger = CSVLogger("logs", name="regressor")

# Instantiate the trainer
trainer = dl.Trainer(
    max_epochs=5000,
    accelerator="auto",
    callbacks=[checkpoint_callback],
    logger=csv_logger,
    log_every_n_steps=5,
    enable_progress_bar=False,
)

In [ ]:
trainer.fit(regressor, train_loader, val_loader)

In [243]:
print(f"The best model is saved at {checkpoint_callback.best_model_path}")

best_sf_model = dl.Regressor.load_from_checkpoint(
    checkpoint_callback.best_model_path
)

The best model is saved at logs/regressor/version_29/checkpoints/epoch=375-step=5264.ckpt


In [ ]:
# Model without trick
best_sf_model = dl.Regressor.load_from_checkpoint(
    "./models/epoch=2557-step=35812.ckpt"
)

In [244]:
# Model with trick
best_sf_model = dl.Regressor.load_from_checkpoint(
    "./models/epoch=375-step=5264.ckpt"
)

In [ ]:
trainer.test(best_sf_model, val_loader)

In [ ]:
best_sf_model.eval()

all_y = []
all_y_hat = []
all_x = []
with torch.no_grad():
    for batch in val_loader:
        x, y = batch


        y_hat = best_sf_model(x)
        all_x.append(x)
        all_y.append(y)
        all_y_hat.append(y_hat)

all_x = torch.cat(all_x, dim=0)
all_y = torch.cat(all_y, dim=0)
all_y_hat = torch.cat(all_y_hat, dim=0)

In [ ]:
plt.scatter(stiffness_range_pa[0] + all_y[:,0]*(stiffness_range_pa[1] - stiffness_range_pa[0]), 
            stiffness_range_pa[0] + all_y_hat[:,0]*(stiffness_range_pa[1] - stiffness_range_pa[0]))

In [ ]:
cut_percentage = 0

import os

if not os.path.exists("3t3_cell_dataset"):
    os.system("git clone https://github.com/DeepTrackAI/3t3_cell_dataset")

from pathlib import Path
import numpy as np

# Load the approach curves
approach_path = Path.cwd() / "3t3_cell_dataset" / "approach"
approach_files = sorted(approach_path.glob("*.npy"), key=lambda f: int(f.stem))
approach_curves = [np.load(f) for f in approach_files]

# Load the retraction curves
retract_path = Path.cwd() / "3t3_cell_dataset" / "retraction"
retract_files = sorted(retract_path.glob("*.npy"), key=lambda f: int(f.stem))
retract_curves = [np.load(f) for f in retract_files]

# Load the labels
label_file = Path.cwd() / "3t3_cell_dataset" / "label.npy"
labels = np.load(label_file)

# Load the contact points
contactpoint_file = Path.cwd() / "3t3_cell_dataset" / "contact_point.npy"
contactpoints = np.load(contactpoint_file)

# Load the contact points
young_modulus_file = Path.cwd() / "3t3_cell_dataset" / "young_modulus.npy"
young_modulus = np.load(young_modulus_file)

# Calculate the length of each curve
approach_curve_lengths = [curve.shape[0] for curve in approach_curves]
retract_curve_lengths = [curve.shape[0] for curve in retract_curves]

# Calculate the minimum length of approach curves and retraction curves
min_approach_curve_length = min(approach_curve_lengths)
min_retract_curve_length = min(retract_curve_lengths)

print(
    f"Minimum approach curve length = {min_approach_curve_length}\n"
    f"Minimum retraction curve length = {min_retract_curve_length}\n"
)


data_exp_x1 = []
data_exp_x2 = []

for i in approach_curves:
    data_exp_x1.append(i[int(i.shape[0]*cut_percentage)::,0])
    data_exp_x2.append(i[int(i.shape[0]*cut_percentage)::,1])

import numpy as np

def resample_to_same_length(data_list, target_len):
    
    resampled = []

    for data in data_list:
        data = np.asarray(data)
        original_len = len(data)

        x_old = np.linspace(0, 1, original_len)

        x_new = np.linspace(0, 1, target_len)

        y_new = np.interp(x_new, x_old, data)

        resampled.append(y_new)

    return resampled

teste1 = resample_to_same_length(data_exp_x1, 1500)
teste2 = resample_to_same_length(data_exp_x2, 1500)

teste1 = np.array(teste1)
teste2 = np.array(teste2)

teste1 = teste1 - teste1.min(axis=1).reshape(-1,1)

data_exp_x1_ = []
data_exp_x2_ = []

for indx in range(len(teste1)):
    cut_percentage = cp_pred_real[indx]
    i = np.where(abs(teste1[indx]-cut_percentage)==abs(teste1[indx]-cut_percentage).min())[0][0]
    data_exp_x1_.append(teste1[indx, i::])
    data_exp_x2_.append(teste2[indx, i::])


teste1 = resample_to_same_length(data_exp_x1_, 1500)
teste2 = resample_to_same_length(data_exp_x2_, 1500)

teste1 = np.array(teste1)
teste2 = np.array(teste2)




data_xe_r = np.hstack([teste1, teste2])
data_ye_r = np.vstack([young_modulus]).T
data_ye_r[:,0] = (data_ye_r[:,0] - stiffness_range_pa[0])/(stiffness_range_pa[1] - stiffness_range_pa[0])

real_dataset = TensorDataset(
    torch.tensor(data_xe_r, dtype=torch.float32),
    torch.tensor(data_ye_r, dtype=torch.float32)
)


exp_loader = DataLoader(real_dataset, batch_size=data_xe_r.shape[0])


In [ ]:
trainer.test(best_sf_model, exp_loader)

In [ ]:
best_sf_model.eval()

all_y = []
all_y_hat = []
all_x = []
with torch.no_grad():
    for batch in exp_loader:
        x, y = batch


        y_hat = best_sf_model(x)
        all_x.append(x)
        all_y.append(y)
        all_y_hat.append(y_hat)

all_x = torch.cat(all_x, dim=0)
all_y = torch.cat(all_y, dim=0)
all_y_hat = torch.cat(all_y_hat, dim=0)

all_x = all_x.cpu().numpy()
all_y = all_y.cpu().numpy()
all_y_hat = all_y_hat.cpu().numpy()

In [ ]:
pre_y = ((stiffness_range_pa[0] + all_y[:,0]*(stiffness_range_pa[1] - stiffness_range_pa[0]))/1000)[labels==1]

In [ ]:
pre_y_hat = ((stiffness_range_pa[0] + all_y_hat[:,0]*(stiffness_range_pa[1] - stiffness_range_pa[0]))/1000)[labels==1]

In [ ]:
plt.scatter(pre_y[pre_y<5], pre_y_hat[pre_y<5])
plt.plot([0,5], [0,5], c="r")
plt.xlabel("True Kpa")
plt.ylabel("Pred Kpa")

In [ ]:
root_mean_squared_error(pre_y[pre_y<5], pre_y_hat[pre_y<5])

In [ ]:
sf_pred = ((stiffness_range_pa[0] + all_y[:,0]*(stiffness_range_pa[1] - stiffness_range_pa[0]))).copy()

## widget

In [ ]:
cut_percentage = 0


import os

if not os.path.exists("3t3_cell_dataset"):
    os.system("git clone https://github.com/DeepTrackAI/3t3_cell_dataset")

from pathlib import Path
import numpy as np

# Load the approach curves
approach_path = Path.cwd() / "3t3_cell_dataset" / "approach"
approach_files = sorted(approach_path.glob("*.npy"), key=lambda f: int(f.stem))
approach_curves = [np.load(f) for f in approach_files]

# Load the retraction curves
retract_path = Path.cwd() / "3t3_cell_dataset" / "retraction"
retract_files = sorted(retract_path.glob("*.npy"), key=lambda f: int(f.stem))
retract_curves = [np.load(f) for f in retract_files]

# Load the labels
label_file = Path.cwd() / "3t3_cell_dataset" / "label.npy"
labels = np.load(label_file)

# Load the contact points
contactpoint_file = Path.cwd() / "3t3_cell_dataset" / "contact_point.npy"
contactpoints = np.load(contactpoint_file)

# Load the contact points
young_modulus_file = Path.cwd() / "3t3_cell_dataset" / "young_modulus.npy"
young_modulus = np.load(young_modulus_file)

# Calculate the length of each curve
approach_curve_lengths = [curve.shape[0] for curve in approach_curves]
retract_curve_lengths = [curve.shape[0] for curve in retract_curves]

# Calculate the minimum length of approach curves and retraction curves
min_approach_curve_length = min(approach_curve_lengths)
min_retract_curve_length = min(retract_curve_lengths)

print(
    f"Minimum approach curve length = {min_approach_curve_length}\n"
    f"Minimum retraction curve length = {min_retract_curve_length}\n"
)


data_exp_x1 = []
data_exp_x2 = []

for i in approach_curves:
    data_exp_x1.append(i[int(i.shape[0]*cut_percentage)::,0])
    data_exp_x2.append(i[int(i.shape[0]*cut_percentage)::,1])

import numpy as np

def resample_to_same_length(data_list, target_len):
    
    resampled = []

    for data in data_list:
        data = np.asarray(data)
        original_len = len(data)

        x_old = np.linspace(0, 1, original_len)

        x_new = np.linspace(0, 1, target_len)

        y_new = np.interp(x_new, x_old, data)

        resampled.append(y_new)

    return resampled

teste1 = resample_to_same_length(data_exp_x1, 1500)
teste2 = resample_to_same_length(data_exp_x2, 1500)

teste1 = np.array(teste1)
teste2 = np.array(teste2)

contactpoints =  contactpoints - teste1.min(axis=1)
teste1 = teste1 - teste1.min(axis=1).reshape(-1,1)
contactpoints =  contactpoints/teste1.max(axis=1)
data_xe_r = np.hstack([teste1, teste2])
data_ye_r = np.vstack([contactpoints, young_modulus]).T
data_ye_r[:,1] = (data_ye_r[:,1] - stiffness_range_pa[0])/(stiffness_range_pa[1] - stiffness_range_pa[0])


In [ ]:
all_x_rec = data_xe_r[labels==1][pre_y<5]

all_y_rec = data_ye_r[labels==1][pre_y<5]

all_y_rec[:,0] = all_y_rec[:,0]*all_x_rec[:,0:1500].max(axis=1)
all_y_rec[:,1] = stiffness_range_pa[0] + all_y_rec[:,1]*(stiffness_range_pa[1] - stiffness_range_pa[0])

all_y_hat_rec = np.vstack([cp_pred_real, sf_pred]).T
all_y_hat_rec = all_y_hat_rec[labels==1][pre_y<5]

In [ ]:
approach_force_ml_list = []
approach_force_gt_list = []

for indx in range(all_x_rec.shape[0]):
    
    approach_axis_um = all_x_rec[indx][0:1500]
    measure_force_pn = all_x_rec[indx][1500::]
    
    contact_point_um_gt = all_y_rec[indx][0]
    stiffness_pa_gt = all_y_rec[indx][1]
    
    contact_point_um_ml = all_y_hat_rec[indx][0]
    stiffness_pa_ml = all_y_hat_rec[indx][1]    

    
    approach_indent = np.maximum(
    contact_point_um_ml/1e6 - approach_axis_um/1e6, 0.0
    )
    approach_force_ml = contact_model.force(
        cantilever_shape=config.tip_shape,
        indentation_m=approach_indent,
        stiffness_pa=stiffness_pa_ml,
    )
    

    
    approach_force_ml_list.append(approach_force_ml*1e9)
    
    approach_indent = np.maximum(
    contact_point_um_gt/1e6 - approach_axis_um/1e6, 0.0
    )
    approach_force_gt = contact_model.force(
        cantilever_shape=config.tip_shape,
        indentation_m=approach_indent,
        stiffness_pa=stiffness_pa_gt,
    )

    approach_force_gt_list.append(approach_force_gt*1e9)

    #plt.plot(approach_axis_um, measure_force_pn)
    #plt.axvline(contact_point_um_ml, c="r")
    #plt.axvline(contact_point_um_gt, c="g")
    #plt.plot(approach_axis_um, approach_force_ml*1e9, label="ML", c="r")
    #plt.plot(approach_axis_um, approach_force_gt*1e9, label="GT", c="g")
    #plt.legend()
    #plt.show()

In [ ]:
approach_force_ml_list = np.array(approach_force_ml_list)
approach_force_gt_list = np.array(approach_force_gt_list)
reconst_loss_ml = ((approach_force_ml_list - all_x_rec[:,1500:])**2).mean(axis=1)
reconst_loss_gt = ((approach_force_gt_list - all_x_rec[:,1500:])**2).mean(axis=1)

In [ ]:
import numpy as np
import plotly.graph_objects as go

all_diff = reconst_loss_ml

# Sort by reconstruction loss
sorted_idx = np.argsort(all_diff)[::-1]
y_list_sort = [approach_force_ml_list[i] for i in sorted_idx]
y_hat_list_sort = [approach_force_gt_list[i] for i in sorted_idx]
x_list_sort = [all_x_rec[i] for i in sorted_idx]
cp_ml_sort = [all_y_hat_rec[i][0] for i in sorted_idx]
cp_gt_sort = [all_y_rec[i][0] for i in sorted_idx]

diff_sort = [all_diff[i] for i in sorted_idx]

# Frames Construction
frames = []
for i in range(len(y_list_sort)):
    frames.append(
        go.Frame(
            data=[
                go.Scatter(x= x_list_sort[i][0:1500], y=x_list_sort[i][1500::], mode="lines", name="Raw Data"),
                go.Scatter(x= x_list_sort[i][0:1500], y=y_list_sort[i],mode="lines", name="ML"),
                go.Scatter(x= x_list_sort[i][0:1500], y=y_hat_list_sort[i],mode="lines", name="GT"),
            ],
            name=str(i),
            layout=go.Layout(title=f"Reconstruction Loss={diff_sort[i]:.4f}",
                    shapes=[dict(type="line",x0=cp_ml_sort[i],x1=cp_ml_sort[i],y0=0,y1=1,yref="paper",line=dict(color="red",width=2,dash="dash")),
                            dict(type="line",x0=cp_gt_sort[i],x1=cp_gt_sort[i],y0=0,y1=1,yref="paper",line=dict(color="green",width=2,dash="dash"))]),
        )
    )

# Initial Picture
fig = go.Figure(
    data=[
        go.Scatter(x= x_list_sort[0][0:1500], y=x_list_sort[0][1500::], mode="lines", name="Raw Data"),
        go.Scatter(x= x_list_sort[0][0:1500], y=y_list_sort[0], mode="lines", name="ML"),
        go.Scatter(x= x_list_sort[0][0:1500], y=y_hat_list_sort[0], mode="lines", name="GT"),
    ],
    layout=go.Layout(title=f"Reconstruction Loss={diff_sort[0]:.4f}",
                    shapes=[dict(type="line",x0=cp_ml_sort[0],x1=cp_ml_sort[0],y0=0,y1=1,yref="paper",line=dict(color="red",width=2,dash="dash")),
                            dict(type="line",x0=cp_gt_sort[0],x1=cp_gt_sort[0],y0=0,y1=1,yref="paper",line=dict(color="green",width=2,dash="dash"))]),
    frames=frames,
)

# Slider
sliders = [
    {
        "steps": [
            {
                "args": [
                    [str(i)],
                    {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"},
                ],
                "label": f"{i}",
                "method": "animate",
            }
            for i in range(len(y_list_sort))
        ],
        "currentvalue": {"prefix": "Index: "},
    }
]

fig.update_layout(sliders=sliders, autosize=False, width=700, height=500)

fig.show()